# 🏷️ Restaurant Text Classification & Sentiment Analysis
### LA Luxury Restaurant Recommendation System — Phase 3

**Purpose:** Enrich the restaurant dataset with four layers of AI-powered classification:

| Layer | Column | Description |
|-------|--------|-------------|
| 1 | `simple_cuisine_group` | 41 raw cuisine types → 11 clean groups (rule-based) |
| 2 | `dining_format` | Infer Tasting Menu / A La Carte / Full Service from description (rule-based) |
| 3 | `predicted_occasion` | Zero-shot LLM: what occasion is this restaurant best for? |
| 4 | `predicted_vibe` | Zero-shot LLM: what is the dominant dining vibe/energy? |

**Input:**  `cleaned_restaurants_final.csv` + `tagged_restaurant_descriptions.txt`  
**Output:** `restaurants_with_classifications.csv` — fully enriched dataset for the Gradio UI

---
**Pipeline Overview:**
```
cleaned_restaurants_final.csv
        │
        ├── Rule-based mapping ──► simple_cuisine_group  (e.g. "Japanese", "Italian")
        ├── Keyword inference ──► dining_format          (e.g. "Tasting Menu / Omakase")
        │
        ▼
  tagged_restaurant_descriptions.txt  (one restaurant per line)
        │
        ├── Zero-shot LLM ──► predicted_occasion         (e.g. "Special Occasion", "Date Night")
        └── Zero-shot LLM ──► predicted_vibe             (e.g. "Intimate", "Lively")
        │
        ▼
  Accuracy check on known labels
        │
        ▼
  restaurants_with_classifications.csv
```

## 📦 Cell 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
from tqdm import tqdm

import torch
from transformers import pipeline

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 220)

# Check device availability for the model
if torch.cuda.is_available():
    DEVICE = 0          # NVIDIA GPU
    device_label = f"GPU — {torch.cuda.get_device_name(0)}"
elif torch.backends.mps.is_available():
    DEVICE = "mps"      # Apple Silicon
    device_label = "Apple Silicon MPS"
else:
    DEVICE = "cpu"      # CPU fallback
    device_label = "CPU"

print(f"✅ Libraries loaded.")
print(f"   PyTorch version  : {torch.__version__}")
print(f"   Inference device : {device_label}")
print(f"   Note: CPU is fine — 94 restaurants classifies in ~2 minutes per task.")

✅ Libraries loaded.
   PyTorch version  : 2.10.0+cpu
   Inference device : CPU
   Note: CPU is fine — 71 restaurants classifies in ~2 minutes per task.


## 📂 Cell 2 — Load the Restaurant Dataset

> **Note:** Place `cleaned_restaurants_final.csv` in the same folder as this notebook,  
> or update `CSV_PATH` below to point to its location.

In [2]:
CSV_PATH = "../data/cleaned_restaurants_final.csv"
TXT_PATH = "../data/tagged_restaurant_descriptions.txt"

restaurants = pd.read_csv(CSV_PATH)

print(f"✅ Dataset loaded: {len(restaurants)} restaurants, {len(restaurants.columns)} columns")
print(f"\nColumns: {restaurants.columns.tolist()}")
print(f"\nMichelin breakdown:")
print(restaurants['Michelin-Guide'].value_counts().to_string())

✅ Dataset loaded: 94 restaurants, 15 columns

Columns: ['Name', 'Location', 'Description', 'Address', 'Telephone Number', 'Price', 'Cuisine Type', 'Dining Atmosphere', 'Sky-High Rooftop', 'Michelin-Guide', 'Customer Ratings', 'Operation Hours', 'Reservations', 'Dress Code', 'restaurant_metadata']

Michelin breakdown:
Michelin-Guide
No                   42
1-Star               21
Michelin-Selected    15
Bib-Gourmand         10
2-Star                4
3-Star                2


## 👀 Cell 3 — Preview Existing Cuisine Types

The dataset has 41 distinct `Cuisine Type` values — far too granular for clean filtering in the UI.  
We'll map them down to 11 meaningful groups.

In [3]:
print(f"=== CURRENT CUISINE TYPES ({restaurants['Cuisine Type'].nunique()} unique values) ===")
print(restaurants['Cuisine Type'].value_counts().to_string())

print(f"\n=== CURRENT DINING ATMOSPHERE ({restaurants['Dining Atmosphere'].nunique()} unique values) ===")
print(restaurants['Dining Atmosphere'].value_counts().to_string())

=== CURRENT CUISINE TYPES (59 unique values) ===
Cuisine Type
Italian                               7
Steakhouse                            5
Japanese Omakase                      4
Contemporary American                 4
Japanese Kaiseki                      3
Japanese / Sushi                      3
Contemporary                          3
Japanese                              3
Asian / Taiwanese                     3
Korean Contemporary                   2
Californian                           2
Mexican                               2
Steakhouse / Seafood                  2
Italian / Modernist                   2
Indian                                2
Korean                                2
Japanese / Peruvian                   2
New American                          2
Spanish Modernist                     1
Seafood / Contemporary                1
French / Contemporary American        1
Contemporary American / Innovative    1
French                                1
Mexican / Seafood 

---
## 🗺️ LAYER 1 — Cuisine Group Mapping (Rule-Based)

Maps all 41 raw `Cuisine Type` values to 11 clean `simple_cuisine_group` labels.  
This is a deterministic lookup — no model inference needed, just a carefully  
designed dictionary built from auditing every entry in the CSV.

## 🗂️ Cell 4 — Define the Cuisine Group Mapping

In [4]:
# Maps all 41 Cuisine Type values → 11 clean cuisine groups
# Groups are designed for Gradio UI filter dropdowns

CUISINE_GROUP_MAP = {
    # ── Japanese (14 restaurants) ──────────────────────────────────
    'Japanese':                         'Japanese',
    'Japanese / Sushi':                 'Japanese',
    'Japanese Omakase':                 'Japanese',
    'Japanese Kaiseki':                 'Japanese',
    'Teppanyaki':                       'Japanese',

    # ── Japanese Fusion (3 restaurants) ───────────────────────────
    'Japanese / Peruvian':              'Japanese Fusion',
    'Japanese / Steakhouse':            'Japanese Fusion',

    # ── Italian (10 restaurants) ───────────────────────────────────
    'Italian':                          'Italian',
    'Italian / Tuscan':                 'Italian',
    'Italian / Modernist':              'Italian',
    'Italian-American':                 'Italian',
    'Pizza':                            'Italian',

    # ── Steakhouse (10 restaurants) ───────────────────────────────
    'Steakhouse':                       'Steakhouse',
    'American Steakhouse':              'Steakhouse',
    'Brazilian Steakhouse':             'Steakhouse',
    'Steakhouse / Seafood':             'Steakhouse',
    'Steakhouse / Sushi':               'Steakhouse',
    'Barbecue':                         'Steakhouse',

    # ── Contemporary American (11 restaurants) ────────────────────
    'Contemporary':                     'Contemporary American',
    'Contemporary American':            'Contemporary American',
    'Contemporary American / Innovative': 'Contemporary American',
    'American':                         'Contemporary American',
    'Californian':                      'Contemporary American',
    'Global / Contemporary':            'Contemporary American',

    # ── French (4 restaurants) ────────────────────────────────────
    'French':                           'French',
    'French / Contemporary American':   'French',
    'French / Mediterranean':           'French',
    'Mediterranean / French-Californian': 'French',

    # ── Seafood (3 restaurants) ───────────────────────────────────
    'Seafood / Contemporary':           'Seafood',
    'Seafood / Steakhouse':             'Seafood',
    'Mexican / Seafood':                'Seafood',

    # ── Asian Fusion (11 restaurants) ─────────────────────────────
    'Asian':                            'Asian Fusion',
    'Asian / Taiwanese':                'Asian Fusion',
    'Korean':                           'Asian Fusion',
    'Korean Contemporary':              'Asian Fusion',
    'Malaysian':                        'Asian Fusion',
    'Indian':                           'Asian Fusion',

    # ── Mexican / Latin (2 restaurants) ──────────────────────────
    'Mexican':                          'Mexican / Latin',

    # ── Spanish (2 restaurants) ───────────────────────────────────
    'Spanish':                          'Spanish',
    'Spanish Modernist':                'Spanish',

    # ── American Casual (1 restaurant) ───────────────────────────
    'Deli':                             'American Casual',
}

print(f"✅ Cuisine group mapping defined: {len(CUISINE_GROUP_MAP)} entries → 11 groups")

# Validate: every existing Cuisine Type is covered
unmapped = set(restaurants['Cuisine Type'].unique()) - set(CUISINE_GROUP_MAP.keys())
if unmapped:
    print(f"⚠️  Unmapped cuisine types (add to CUISINE_GROUP_MAP): {unmapped}")
else:
    print("✅ All cuisine types are mapped — no gaps.")

✅ Cuisine group mapping defined: 41 entries → 11 groups
⚠️  Unmapped cuisine types (add to CUISINE_GROUP_MAP): {'Persian', 'Nordic-Californian', 'Italian / Milanese', 'Pan Asian', 'Asian / Indonesian', 'Asian / New American', 'Seafood', 'Asian Fusion', 'Asian / Vietnamese', 'Turkish', 'New American / Latin', 'Japanese /  New American', 'Contemporary American / Seafood', 'New Mexican', 'New American', 'Japanese Izakaya', 'Japanese / Brazilian', 'New American / Steakhouse'}


## ✅ Cell 5 — Apply Cuisine Group Mapping

In [5]:
restaurants['simple_cuisine_group'] = restaurants['Cuisine Type'].map(CUISINE_GROUP_MAP)

print("=== CUISINE GROUP DISTRIBUTION ===")
group_counts = restaurants['simple_cuisine_group'].value_counts()
for group, count in group_counts.items():
    bar = '█' * count
    print(f"  {group:<25} | {bar} ({count})")

# Check for any null mappings
null_groups = restaurants[restaurants['simple_cuisine_group'].isna()]
if null_groups.empty:
    print("\n✅ All 71 restaurants have a cuisine group assigned.")
else:
    print(f"\n⚠️  {len(null_groups)} restaurants have no group:")
    print(null_groups[['Name', 'Cuisine Type']].to_string())

=== CUISINE GROUP DISTRIBUTION ===
  Japanese                  | ██████████████ (14)
  Contemporary American     | ████████████ (12)
  Italian                   | ████████████ (12)
  Asian Fusion              | ███████████ (11)
  Steakhouse                | ███████████ (11)
  French                    | ████ (4)
  Seafood                   | ███ (3)
  Japanese Fusion           | ███ (3)
  Spanish                   | ██ (2)
  Mexican / Latin           | ██ (2)
  American Casual           | █ (1)

⚠️  19 restaurants have no group:
                      Name                     Cuisine Type
71             Sushi Samba             Japanese / Brazilian
73                  Lielle               Nordic-Californian
75                    Padi               Asian / Indonesian
76                   Perse                          Persian
77            Bar di Bello               Italian / Milanese
78             The Lobster                          Seafood
81   SALT Restaurant & Bar  Contemporary Amer

## 🔍 Cell 6 — Verify Cuisine Grouping (Spot Check)

In [6]:
print("=== FULL CUISINE TYPE → GROUP MAPPING VERIFICATION ===")
print(f"{'Restaurant':<35} {'Cuisine Type':<35} {'Group'}")
print("-" * 95)
for _, row in restaurants[['Name', 'Cuisine Type', 'simple_cuisine_group']].iterrows():
    print(f"  {row['Name']:<33} {row['Cuisine Type']:<33} → {row['simple_cuisine_group']}")

=== FULL CUISINE TYPE → GROUP MAPPING VERIFICATION ===
Restaurant                          Cuisine Type                        Group
-----------------------------------------------------------------------------------------------
  Somni                             Spanish Modernist                 → Spanish
  Providence                        Seafood / Contemporary            → Seafood
  Hayato                            Japanese Kaiseki                  → Japanese
  n/naka                            Japanese Kaiseki                  → Japanese
  Melisse                           French / Contemporary American    → French
  Vespertine                        Contemporary American / Innovative → Contemporary American
  Sushi Kaneyoshi                   Japanese / Sushi                  → Japanese
  Restaurant Ki                     Korean Contemporary               → Asian Fusion
  715 Sushi                         Japanese / Sushi                  → Japanese
  Orsa & Winston            

---
## 🍽️ LAYER 2 — Dining Format Classification (Rule-Based)

Infers the dining format from the `Description` column using keyword matching.  
This captures how guests actually experience the restaurant, which is critical  
for recommending based on dining style (tasting menu vs ordering freely).

**Format Labels:**
| Label | Meaning | Example |
|-------|---------|---------|  
| `Tasting Menu / Omakase` | Chef-driven fixed course progression | Hayato, n/naka, Heritage |
| `A La Carte / Casual` | Order freely from a menu | Langer's, Parks BBQ, Holbox |
| `Full Service` | Formal table service, mix of formats | CUT, Bestia, Osteria Mozza |

## 🎯 Cell 7 — Define Dining Format Inference Logic

In [7]:
# Keywords that signal a fixed tasting menu or omakase format
TASTING_KEYWORDS = [
    'tasting menu', 'omakase', 'kaiseki', 'multi-course',
    'chef\'s choice', 'fixed course', 'prix-fixe', 'prix fixe'
]

# Keywords that signal casual, à la carte ordering
ALACARTE_KEYWORDS = [
    'a la carte', 'counter', 'food hall', 'deli', 'tacos',
    'barbecue', 'bbq', 'fast', 'cafeteria'
]


def infer_dining_format(row: pd.Series) -> str:
    """
    Infers dining format from Description text and Dining Atmosphere.

    Priority order:
    1. Tasting Menu / Omakase — if description contains fixed-course keywords
    2. A La Carte / Casual    — if description signals casual ordering, or atmosphere is Casual
    3. Full Service           — default for formal table-service restaurants
    """
    desc = str(row['Description']).lower()
    atmosphere = str(row['Dining Atmosphere']).lower()

    # Check for tasting menu signals first (highest priority)
    if any(kw in desc for kw in TASTING_KEYWORDS):
        return 'Tasting Menu / Omakase'

    # Check for a la carte / casual signals
    if any(kw in desc for kw in ALACARTE_KEYWORDS):
        return 'A La Carte / Casual'

    # Atmosphere-based fallback for casual spots
    if 'casual' in atmosphere:
        return 'A La Carte / Casual'

    # Default: full-service table dining
    return 'Full Service'


restaurants['dining_format'] = restaurants.apply(infer_dining_format, axis=1)

print("=== DINING FORMAT DISTRIBUTION ===")
print(restaurants['dining_format'].value_counts().to_string())

print("\n=== TASTING MENU / OMAKASE RESTAURANTS ===")
tasting = restaurants[restaurants['dining_format'] == 'Tasting Menu / Omakase']
print(tasting[['Name', 'Michelin-Guide', 'Price', 'dining_format']].to_string(index=True))

=== DINING FORMAT DISTRIBUTION ===
dining_format
Full Service              37
A La Carte / Casual       37
Tasting Menu / Omakase    20

=== TASTING MENU / OMAKASE RESTAURANTS ===
                  Name     Michelin-Guide  Price           dining_format
0                Somni             3-Star  $$$$$  Tasting Menu / Omakase
2               Hayato             2-Star   $$$$  Tasting Menu / Omakase
3               n/naka             2-Star   $$$$  Tasting Menu / Omakase
4              Melisse             2-Star   $$$$  Tasting Menu / Omakase
6      Sushi Kaneyoshi             1-Star   $$$$  Tasting Menu / Omakase
7        Restaurant Ki             1-Star   $$$$  Tasting Menu / Omakase
8            715 Sushi             1-Star   $$$$  Tasting Menu / Omakase
9       Orsa & Winston             1-Star   $$$$  Tasting Menu / Omakase
11            Morihiro             1-Star   $$$$  Tasting Menu / Omakase
12                Kato             1-Star   $$$$  Tasting Menu / Omakase
15            Her

## 🔍 Cell 8 — Dining Format Full Review

In [8]:
print("=== ALL RESTAURANTS — DINING FORMAT ASSIGNMENTS ===")
print(f"{'Restaurant':<35} {'Atmosphere':<28} {'Format'}")
print("-" * 95)
for _, row in restaurants[['Name', 'Dining Atmosphere', 'dining_format']].iterrows():
    print(f"  {row['Name']:<33} {row['Dining Atmosphere']:<26} → {row['dining_format']}")

=== ALL RESTAURANTS — DINING FORMAT ASSIGNMENTS ===
Restaurant                          Atmosphere                   Format
-----------------------------------------------------------------------------------------------
  Somni                             Fine-Dining                → Tasting Menu / Omakase
  Providence                        Fine-Dining                → Full Service
  Hayato                            Fine-Dining                → Tasting Menu / Omakase
  n/naka                            Fine-Dining                → Tasting Menu / Omakase
  Melisse                           Fine-Dining                → Tasting Menu / Omakase
  Vespertine                        Fine-Dining                → Full Service
  Sushi Kaneyoshi                   Fine-Dining                → Tasting Menu / Omakase
  Restaurant Ki                     Fine-Dining                → Tasting Menu / Omakase
  715 Sushi                         Fine-Dining                → Tasting Menu / Omakase
  Orsa &

---
## 🤖 LAYER 3 — Occasion Classification (Zero-Shot LLM)

Uses `facebook/bart-large-mnli` zero-shot classification to predict the **ideal occasion**  
for each restaurant based on its `restaurant_metadata` text.  

**Why zero-shot?** Our 94-restaurant dataset has no labeled "occasion" column, so we  
can't train a classifier. Zero-shot models reason about candidate labels without  
any training examples — perfect for a curated niche dataset like this.

**Occasion Labels:**
| Label | Meaning |
|-------|---------|
| `Special Occasion` | Milestone celebrations — anniversaries, birthdays, proposals |
| `Date Night` | Romantic, intimate, memorable evening for two |
| `Business Dining` | Professional, impressive, good for client entertainment |
| `Casual Night Out` | Relaxed, fun, no dress code stress |
| `Foodie Adventure` | Bucket-list dining, experimental, chef-driven |
| `Group Celebration` | Lively, shareable, good for parties |

## ⚙️ Cell 9 — Load Zero-Shot Classification Model

In [9]:
ZS_MODEL = "facebook/bart-large-mnli"

print(f"⏳ Loading zero-shot classification model: {ZS_MODEL}")
print("   First run downloads ~1.6GB model — subsequent runs load from HuggingFace cache.")
print()

classifier = pipeline(
    "zero-shot-classification",
    model=ZS_MODEL,
    device=DEVICE
)

print(f"✅ Zero-shot model loaded on device: {DEVICE}")

# Quick smoke test
test_text = "An intimate 7-seat kaiseki restaurant with multi-course tasting menu and rare seasonal ingredients."
test_result = classifier(test_text, candidate_labels=["Special Occasion", "Casual Night Out"])
print(f"\nSmoke test: '{test_result['labels'][0]}' — confidence: {test_result['scores'][0]:.3f} ✅")

⏳ Loading zero-shot classification model: facebook/bart-large-mnli
   First run downloads ~1.6GB model — subsequent runs load from HuggingFace cache.



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Zero-shot model loaded on device: cpu

Smoke test: 'Special Occasion' — confidence: 0.786 ✅


## 🛠️ Cell 10 — Define Occasion Labels & Classification Helper

In [10]:
# Occasion labels — designed to cover all key use cases in the recommendation UI
OCCASION_LABELS = [
    "Special Occasion",     # anniversaries, proposals, milestone birthdays
    "Date Night",           # intimate, romantic, memorable evening for two
    "Business Dining",      # professional, impressive, client entertainment
    "Casual Night Out",     # relaxed, fun, affordable, no fuss
    "Foodie Adventure",     # tasting menus, experimental, chef-driven
    "Group Celebration",    # lively, shareable plates, good for parties
]


def classify_occasion(text: str, labels: list) -> str:
    """
    Uses zero-shot classification to predict the best dining occasion
    for a restaurant based on its metadata text.

    Parameters
    ----------
    text   : The restaurant_metadata string for this restaurant
    labels : List of candidate occasion label strings

    Returns
    -------
    str : The highest-scoring occasion label
    """
    if not text or not labels:
        return "Unknown"

    result = classifier(text, candidate_labels=labels)
    best_index = np.argmax(result["scores"])
    return result["labels"][best_index]


# Test on a sample restaurant
test_meta = restaurants.loc[0, 'restaurant_metadata']  # Somni
test_occasion = classify_occasion(test_meta, OCCASION_LABELS)
print(f"Test — {restaurants.loc[0, 'Name']}: predicted occasion = '{test_occasion}'")
print(f"\n✅ classify_occasion() is ready.")

Test — Somni: predicted occasion = 'Special Occasion'

✅ classify_occasion() is ready.


## 🔄 Cell 11 — Run Occasion Classification on All 71 Restaurants

In [11]:
print("⏳ Classifying dining occasion for all 71 restaurants...")
print(f"   Labels: {OCCASION_LABELS}")
print()

occasion_predictions = []
occasion_scores = []

for i in tqdm(range(len(restaurants)), desc="Occasion Classification"):
    metadata = restaurants.loc[i, 'restaurant_metadata']
    result = classifier(metadata, candidate_labels=OCCASION_LABELS)
    best_idx = np.argmax(result["scores"])
    occasion_predictions.append(result["labels"][best_idx])
    occasion_scores.append(round(result["scores"][best_idx], 4))

restaurants['predicted_occasion'] = occasion_predictions
restaurants['occasion_confidence'] = occasion_scores

print(f"\n✅ Occasion classification complete.")
print(f"\n=== PREDICTED OCCASION DISTRIBUTION ===")
print(restaurants['predicted_occasion'].value_counts().to_string())
print(f"\nAverage confidence: {np.mean(occasion_scores):.3f}")

⏳ Classifying dining occasion for all 71 restaurants...
   Labels: ['Special Occasion', 'Date Night', 'Business Dining', 'Casual Night Out', 'Foodie Adventure', 'Group Celebration']



Occasion Classification: 100%|██████████| 94/94 [02:16<00:00,  1.45s/it]


✅ Occasion classification complete.

=== PREDICTED OCCASION DISTRIBUTION ===
predicted_occasion
Special Occasion    69
Foodie Adventure    16
Casual Night Out     8
Business Dining      1

Average confidence: 0.359


## 📋 Cell 12 — Review Occasion Predictions (Full Table)

In [12]:
print("=== OCCASION PREDICTIONS — ALL RESTAURANTS ===")
review_cols = ['Name', 'Michelin-Guide', 'Price', 'Dining Atmosphere', 
               'predicted_occasion', 'occasion_confidence']
print(restaurants[review_cols].to_string(index=True))

=== OCCASION PREDICTIONS — ALL RESTAURANTS ===
                              Name     Michelin-Guide  Price        Dining Atmosphere predicted_occasion  occasion_confidence
0                            Somni             3-Star  $$$$$              Fine-Dining   Special Occasion               0.3330
1                       Providence             3-Star   $$$$              Fine-Dining   Special Occasion               0.4971
2                           Hayato             2-Star   $$$$              Fine-Dining   Special Occasion               0.3729
3                           n/naka             2-Star   $$$$              Fine-Dining   Special Occasion               0.2660
4                          Melisse             2-Star   $$$$              Fine-Dining   Special Occasion               0.3619
5                       Vespertine             2-Star   $$$$              Fine-Dining   Special Occasion               0.4762
6                  Sushi Kaneyoshi             1-Star   $$$$           

## 🧪 Cell 13 — Accuracy Check: Occasion vs Known Labels

We don't have ground-truth occasion labels, but we can validate predictions  
against known characteristics. The spot-check below tests logical expectations  
that should hold true based on what we know about the data.

In [13]:
print("=== OCCASION ACCURACY SPOT-CHECK ===")
print("Checking logical expectations against predicted occasions:\n")

# Expectation 1: 3-Star Michelin restaurants should lean toward Special Occasion or Foodie Adventure
three_star = restaurants[restaurants['Michelin-Guide'] == '3-Star']
expected_3star = {'Special Occasion', 'Foodie Adventure', 'Date Night'}
pass_3star = all(row['predicted_occasion'] in expected_3star for _, row in three_star.iterrows())
for _, row in three_star.iterrows():
    status = "✅" if row['predicted_occasion'] in expected_3star else "⚠️ "
    print(f"  {status} {row['Name']:<30} ({row['Michelin-Guide']}) → {row['predicted_occasion']}")

# Expectation 2: Bib Gourmand should lean toward Casual Night Out or Foodie Adventure
print()
bib = restaurants[restaurants['Michelin-Guide'] == 'Bib-Gourmand']
expected_bib = {'Casual Night Out', 'Foodie Adventure', 'Group Celebration'}
for _, row in bib.iterrows():
    status = "✅" if row['predicted_occasion'] in expected_bib else "⚠️ "
    print(f"  {status} {row['Name']:<30} (Bib-Gourmand) → {row['predicted_occasion']}")

# Expectation 3: Romantic atmosphere should include Date Night or Special Occasion
print()
romantic = restaurants[restaurants['Dining Atmosphere'].str.contains('Romantic', na=False)]
expected_romantic = {'Date Night', 'Special Occasion'}
for _, row in romantic.iterrows():
    status = "✅" if row['predicted_occasion'] in expected_romantic else "⚠️ "
    print(f"  {status} {row['Name']:<30} (Romantic) → {row['predicted_occasion']}")

# Accuracy rate for these spot-checks
all_checks = []
for _, row in three_star.iterrows():
    all_checks.append(row['predicted_occasion'] in expected_3star)
for _, row in bib.iterrows():
    all_checks.append(row['predicted_occasion'] in expected_bib)
for _, row in romantic.iterrows():
    all_checks.append(row['predicted_occasion'] in expected_romantic)

accuracy = sum(all_checks) / len(all_checks)
print(f"\n=== SPOT-CHECK ACCURACY: {accuracy:.1%} ({sum(all_checks)}/{len(all_checks)} passed) ===")

=== OCCASION ACCURACY SPOT-CHECK ===
Checking logical expectations against predicted occasions:

  ✅ Somni                          (3-Star) → Special Occasion
  ✅ Providence                     (3-Star) → Special Occasion

  ✅ Maccheroni Republic            (Bib-Gourmand) → Casual Night Out
  ⚠️  The Factory Kitchen            (Bib-Gourmand) → Special Occasion
  ⚠️  Pizzeria Bianco                (Bib-Gourmand) → Special Occasion
  ✅ Tsubaki                        (Bib-Gourmand) → Casual Night Out
  ⚠️  Langer's                       (Bib-Gourmand) → Special Occasion
  ⚠️  Moo's Craft Barbecue           (Bib-Gourmand) → Special Occasion
  ⚠️  Rasarumah                      (Bib-Gourmand) → Special Occasion
  ⚠️  Komal                          (Bib-Gourmand) → Special Occasion
  ✅ Pine & Crane                   (Bib-Gourmand) → Foodie Adventure
  ✅ Pine & Crane                   (Bib-Gourmand) → Foodie Adventure

  ✅ Heritage                       (Romantic) → Special Occasion
  ✅ Kali

---
## 🎭 LAYER 4 — Dining Vibe Classification (Zero-Shot LLM)

Predicts the dominant **energy and vibe** a diner will experience at each restaurant.  
This is distinct from Occasion (what you're celebrating) and Atmosphere (the décor category).  
Vibe captures the *feeling* of the experience — essential for experience-driven queries  
like "somewhere buzzy and fun" or "cozy and quiet".

**Vibe Labels:**
| Label | Meaning |
|-------|---------|
| `Intimate` | Small, quiet, personal — counter seats, few tables |
| `Lively & Social` | Energetic, buzzy, great for groups, good noise level |
| `Refined & Elegant` | Polished, formal, white-glove service |
| `Hip & Trendy` | Cool crowd, design-forward, Instagram-worthy |
| `Cozy & Relaxed` | Warm, unpretentious, neighbourhood feel |
| `Theatrical & Experiential` | Immersive, dramatic, the dining IS the event |

## 🛠️ Cell 14 — Define Vibe Labels & Classification Helper

In [14]:
VIBE_LABELS = [
    "Intimate",                    # small, personal, counter-seat, quiet
    "Lively & Social",             # energetic, buzzy, great for groups
    "Refined & Elegant",           # polished, formal, white-glove
    "Hip & Trendy",                # cool, design-forward, Instagram-worthy
    "Cozy & Relaxed",              # warm, unpretentious, neighbourhood feel
    "Theatrical & Experiential",   # immersive, dramatic, the dining IS the event
]


def classify_vibe(text: str, labels: list) -> str:
    """
    Uses zero-shot classification to predict the dominant dining vibe
    for a restaurant based on its metadata text.

    Parameters
    ----------
    text   : The restaurant_metadata string for this restaurant
    labels : List of candidate vibe label strings

    Returns
    -------
    str : The highest-scoring vibe label
    """
    if not text or not labels:
        return "Unknown"

    result = classifier(text, candidate_labels=labels)
    best_index = np.argmax(result["scores"])
    return result["labels"][best_index]


# Test on a few known restaurants
test_cases = [
    (0,  "Somni"),       # 14-seat counter, avant-garde Spanish — expect Theatrical or Intimate
    (5,  "Vespertine"),  # sci-fi obelisk — expect Theatrical
    (26, "Maccheroni Republic"),  # trattoria — expect Cozy or Lively
]
print("=== VIBE CLASSIFICATION TESTS ===")
for idx, name in test_cases:
    meta = restaurants.loc[idx, 'restaurant_metadata']
    vibe = classify_vibe(meta, VIBE_LABELS)
    print(f"  {name:<30} → {vibe}")

print(f"\n✅ classify_vibe() is ready.")

=== VIBE CLASSIFICATION TESTS ===
  Somni                          → Refined & Elegant
  Vespertine                     → Hip & Trendy
  Maccheroni Republic            → Cozy & Relaxed

✅ classify_vibe() is ready.


## 🔄 Cell 15 — Run Vibe Classification on All 71 Restaurants

In [15]:
print("⏳ Classifying dining vibe for all 71 restaurants...")
print(f"   Labels: {VIBE_LABELS}")
print()

vibe_predictions = []
vibe_scores = []

for i in tqdm(range(len(restaurants)), desc="Vibe Classification"):
    metadata = restaurants.loc[i, 'restaurant_metadata']
    result = classifier(metadata, candidate_labels=VIBE_LABELS)
    best_idx = np.argmax(result["scores"])
    vibe_predictions.append(result["labels"][best_idx])
    vibe_scores.append(round(result["scores"][best_idx], 4))

restaurants['predicted_vibe'] = vibe_predictions
restaurants['vibe_confidence'] = vibe_scores

print(f"\n✅ Vibe classification complete.")
print(f"\n=== PREDICTED VIBE DISTRIBUTION ===")
print(restaurants['predicted_vibe'].value_counts().to_string())
print(f"\nAverage confidence: {np.mean(vibe_scores):.3f}")

⏳ Classifying dining vibe for all 71 restaurants...
   Labels: ['Intimate', 'Lively & Social', 'Refined & Elegant', 'Hip & Trendy', 'Cozy & Relaxed', 'Theatrical & Experiential']



Vibe Classification: 100%|██████████| 94/94 [02:43<00:00,  1.74s/it]


✅ Vibe classification complete.

=== PREDICTED VIBE DISTRIBUTION ===
predicted_vibe
Refined & Elegant            42
Cozy & Relaxed               18
Hip & Trendy                 17
Intimate                      8
Theatrical & Experiential     6
Lively & Social               3

Average confidence: 0.482


## 📋 Cell 16 — Review Vibe Predictions (Full Table)

In [16]:
print("=== VIBE PREDICTIONS — ALL RESTAURANTS ===")
review_cols = ['Name', 'Michelin-Guide', 'Dining Atmosphere', 
               'predicted_vibe', 'vibe_confidence']
print(restaurants[review_cols].to_string(index=True))

=== VIBE PREDICTIONS — ALL RESTAURANTS ===
                              Name     Michelin-Guide        Dining Atmosphere             predicted_vibe  vibe_confidence
0                            Somni             3-Star              Fine-Dining          Refined & Elegant           0.5821
1                       Providence             3-Star              Fine-Dining               Hip & Trendy           0.5285
2                           Hayato             2-Star              Fine-Dining                   Intimate           0.7631
3                           n/naka             2-Star              Fine-Dining          Refined & Elegant           0.4390
4                          Melisse             2-Star              Fine-Dining          Refined & Elegant           0.3358
5                       Vespertine             2-Star              Fine-Dining               Hip & Trendy           0.4622
6                  Sushi Kaneyoshi             1-Star              Fine-Dining                  

## 🧪 Cell 17 — Accuracy Check: Vibe vs Known Characteristics

In [17]:
print("=== VIBE ACCURACY SPOT-CHECK ===")
print("Checking logical expectations against predicted vibes:\n")

# Expectation 1: Avant-garde / innovative restaurants → Theatrical & Experiential
theatrical_expected = ['Vespertine', 'Somni', 'Meteora']
expected_theatrical = {'Theatrical & Experiential', 'Intimate'}
print("Theatrical/Experiential restaurants:")
for name in theatrical_expected:
    row = restaurants[restaurants['Name'] == name].iloc[0]
    status = "✅" if row['predicted_vibe'] in expected_theatrical else "⚠️ "
    print(f"  {status} {name:<30} → {row['predicted_vibe']}")

# Expectation 2: Bib Gourmand casual spots → Cozy & Relaxed or Lively & Social
print()
bib = restaurants[restaurants['Michelin-Guide'] == 'Bib-Gourmand']
expected_bib_vibe = {'Cozy & Relaxed', 'Lively & Social', 'Hip & Trendy'}
print("Bib Gourmand vibes:")
for _, row in bib.iterrows():
    status = "✅" if row['predicted_vibe'] in expected_bib_vibe else "⚠️ "
    print(f"  {status} {row['Name']:<30} → {row['predicted_vibe']}")

# Expectation 3: 2-Star and 3-Star → Refined & Elegant or Intimate
print()
top_michelin = restaurants[restaurants['Michelin-Guide'].isin(['2-Star', '3-Star'])]
expected_top = {'Refined & Elegant', 'Intimate', 'Theatrical & Experiential'}
print("2-Star / 3-Star vibes:")
for _, row in top_michelin.iterrows():
    status = "✅" if row['predicted_vibe'] in expected_top else "⚠️ "
    print(f"  {status} {row['Name']:<30} ({row['Michelin-Guide']}) → {row['predicted_vibe']}")

# Overall spot-check accuracy
all_vibe_checks = []
for name in theatrical_expected:
    row = restaurants[restaurants['Name'] == name].iloc[0]
    all_vibe_checks.append(row['predicted_vibe'] in expected_theatrical)
for _, row in bib.iterrows():
    all_vibe_checks.append(row['predicted_vibe'] in expected_bib_vibe)
for _, row in top_michelin.iterrows():
    all_vibe_checks.append(row['predicted_vibe'] in expected_top)

vibe_accuracy = sum(all_vibe_checks) / len(all_vibe_checks)
print(f"\n=== SPOT-CHECK ACCURACY: {vibe_accuracy:.1%} ({sum(all_vibe_checks)}/{len(all_vibe_checks)} passed) ===")

=== VIBE ACCURACY SPOT-CHECK ===
Checking logical expectations against predicted vibes:

Theatrical/Experiential restaurants:
  ⚠️  Vespertine                     → Hip & Trendy
  ⚠️  Somni                          → Refined & Elegant
  ⚠️  Meteora                        → Hip & Trendy

Bib Gourmand vibes:
  ✅ Maccheroni Republic            → Cozy & Relaxed
  ⚠️  The Factory Kitchen            → Refined & Elegant
  ✅ Pizzeria Bianco                → Cozy & Relaxed
  ⚠️  Tsubaki                        → Refined & Elegant
  ⚠️  Langer's                       → Refined & Elegant
  ✅ Moo's Craft Barbecue           → Cozy & Relaxed
  ✅ Rasarumah                      → Cozy & Relaxed
  ⚠️  Komal                          → Refined & Elegant
  ✅ Pine & Crane                   → Cozy & Relaxed
  ✅ Pine & Crane                   → Cozy & Relaxed

2-Star / 3-Star vibes:
  ✅ Somni                          (3-Star) → Refined & Elegant
  ⚠️  Providence                     (3-Star) → Hip & Trendy
  ✅

---
## 📊 COMBINED REVIEW — All Four Classification Layers

## 🗂️ Cell 18 — Full Classification Summary Table

In [18]:
print("=== COMPLETE CLASSIFICATION RESULTS — ALL 71 RESTAURANTS ===")
summary_cols = [
    'Name', 'Michelin-Guide', 'Price',
    'simple_cuisine_group', 'dining_format',
    'predicted_occasion', 'predicted_vibe'
]
restaurants[summary_cols]

=== COMPLETE CLASSIFICATION RESULTS — ALL 71 RESTAURANTS ===


,Name,Michelin-Guide,Price,simple_cuisine_group,dining_format,predicted_occasion,predicted_vibe
0,Somni,3-Star,$$$$$,Spanish,Tasting Menu / Omakase,Special Occasion,Refined & Elegant
1,Providence,3-Star,$$$$,Seafood,Full Service,Special Occasion,Hip & Trendy
2,Hayato,2-Star,$$$$,Japanese,Tasting Menu / Omakase,Special Occasion,Intimate
3,n/naka,2-Star,$$$$,Japanese,Tasting Menu / Omakase,Special Occasion,Refined & Elegant
4,Melisse,2-Star,$$$$,French,Tasting Menu / Omakase,Special Occasion,Refined & Elegant
...,...,...,...,...,...,...,...
89,Broken Spanish Comedor,No,$$$,NaN,A La Carte / Casual,Special Occasion,Cozy & Relaxed
90,Night We Met,No,$$$,NaN,A La Carte / Casual,Casual Night Out,Cozy & Relaxed
91,Sora Craft Kitchen,Michelin-Selected,$$$,NaN,A La Carte / Casual,Foodie Adventure,Cozy & Relaxed
92,Delilah,No,$$$,NaN,A La Carte / Casual,Special Occasion,Refined & Elegant


## 📈 Cell 19 — Cross-Tabulation: Cuisine Group × Predicted Occasion

In [19]:
print("=== CUISINE GROUP × PREDICTED OCCASION ===")
ct_occasion = pd.crosstab(
    restaurants['simple_cuisine_group'],
    restaurants['predicted_occasion'],
    margins=True
)
print(ct_occasion.to_string())

=== CUISINE GROUP × PREDICTED OCCASION ===
predicted_occasion     Business Dining  Casual Night Out  Foodie Adventure  Special Occasion  All
simple_cuisine_group                                                                             
American Casual                      0                 0                 0                 1    1
Asian Fusion                         0                 0                 3                 8   11
Contemporary American                0                 0                 0                12   12
French                               0                 0                 0                 4    4
Italian                              0                 1                 1                10   12
Japanese                             0                 2                 1                11   14
Japanese Fusion                      0                 0                 1                 2    3
Mexican / Latin                      0                 0                 0 

## 📈 Cell 20 — Cross-Tabulation: Cuisine Group × Predicted Vibe

In [20]:
print("=== CUISINE GROUP × PREDICTED VIBE ===")
ct_vibe = pd.crosstab(
    restaurants['simple_cuisine_group'],
    restaurants['predicted_vibe'],
    margins=True
)
print(ct_vibe.to_string())

=== CUISINE GROUP × PREDICTED VIBE ===
predicted_vibe         Cozy & Relaxed  Hip & Trendy  Intimate  Lively & Social  Refined & Elegant  Theatrical & Experiential  All
simple_cuisine_group                                                                                                             
American Casual                     0             0         0                0                  1                          0    1
Asian Fusion                        4             2         1                0                  4                          0   11
Contemporary American               0             4         1                1                  4                          2   12
French                              0             0         0                0                  4                          0    4
Italian                             3             3         0                0                  5                          1   12
Japanese                            1             0

## 📈 Cell 21 — Cross-Tabulation: Michelin × Occasion & Vibe

In [21]:
michelin_order = ['3-Star', '2-Star', '1-Star', 'Bib-Gourmand', 'Michelin-Selected', 'No']

print("=== MICHELIN TIER × PREDICTED OCCASION ===")
ct_mich_occ = pd.crosstab(
    pd.Categorical(restaurants['Michelin-Guide'], categories=michelin_order, ordered=True),
    restaurants['predicted_occasion']
)
print(ct_mich_occ.to_string())

print("\n=== MICHELIN TIER × PREDICTED VIBE ===")
ct_mich_vibe = pd.crosstab(
    pd.Categorical(restaurants['Michelin-Guide'], categories=michelin_order, ordered=True),
    restaurants['predicted_vibe']
)
print(ct_mich_vibe.to_string())

=== MICHELIN TIER × PREDICTED OCCASION ===
predicted_occasion  Business Dining  Casual Night Out  Foodie Adventure  Special Occasion
row_0                                                                                    
3-Star                            0                 0                 0                 2
2-Star                            0                 0                 0                 4
1-Star                            1                 0                 2                18
Bib-Gourmand                      0                 2                 2                 6
Michelin-Selected                 0                 1                 2                12
No                                0                 5                10                27

=== MICHELIN TIER × PREDICTED VIBE ===
predicted_vibe     Cozy & Relaxed  Hip & Trendy  Intimate  Lively & Social  Refined & Elegant  Theatrical & Experiential
row_0                                                                       

## 🔍 Cell 22 — Interactive Classification Lookup

Look up the full classification profile for any restaurant by name.

In [22]:
def show_restaurant_profile(name: str) -> None:
    """
    Prints the full classification profile for a single restaurant.
    Accepts partial name matching (case-insensitive).
    """
    matches = restaurants[restaurants['Name'].str.contains(name, case=False, na=False)]
    if matches.empty:
        print(f"⚠️  No restaurant found matching '{name}'")
        print(f"   Available names: {restaurants['Name'].tolist()}")
        return

    for _, row in matches.iterrows():
        print("=" * 58)
        print(f"  🍽️  {row['Name']}")
        print("=" * 58)
        print(f"  Location         : {row['Location']}")
        print(f"  Cuisine Type     : {row['Cuisine Type']}")
        print(f"  Michelin-Guide   : {row['Michelin-Guide']}")
        print(f"  Price            : {row['Price']}")
        print(f"  Customer Rating  : {row['Customer Ratings']}/5")
        print()
        print(f"  ── CLASSIFICATIONS ──────────────────────────")
        print(f"  Cuisine Group    : {row['simple_cuisine_group']}")
        print(f"  Dining Format    : {row['dining_format']}")
        print(f"  Occasion         : {row['predicted_occasion']}  (conf: {row['occasion_confidence']:.3f})")
        print(f"  Vibe             : {row['predicted_vibe']}  (conf: {row['vibe_confidence']:.3f})")
        print()
        print(f"  Description: {row['Description']}")
        print()


# Try a few examples
show_restaurant_profile("Vespertine")
show_restaurant_profile("Holbox")
show_restaurant_profile("71Above")

  🍽️  Vespertine
  Location         : Culver City
  Cuisine Type     : Contemporary American / Innovative
  Michelin-Guide   : 2-Star
  Price            : $$$$
  Customer Rating  : 5.0/5

  ── CLASSIFICATIONS ──────────────────────────
  Cuisine Group    : Contemporary American
  Dining Format    : Full Service
  Occasion         : Special Occasion  (conf: 0.476)
  Vibe             : Hip & Trendy  (conf: 0.462)

  Description: An avant-garde two-Michelin-star dining experience from Chef Jordan Kahn set inside a wavy obelisk architectural marvel with a sci-fi dreamscape atmosphere.

  🍽️  Holbox
  Location         : South Los Angeles
  Cuisine Type     : Mexican / Seafood
  Michelin-Guide   : 1-Star
  Price            : $$
  Customer Rating  : 5.0/5

  ── CLASSIFICATIONS ──────────────────────────
  Cuisine Group    : Seafood
  Dining Format    : A La Carte / Casual
  Occasion         : Business Dining  (conf: 0.253)
  Vibe             : Cozy & Relaxed  (conf: 0.662)

  Description: A Y

## 🔁 Cell 23 — Enrich the restaurant_metadata with New Classification Labels

Updates the `restaurant_metadata` column to include the new `predicted_occasion`  
and `predicted_vibe` labels. This ensures the Chroma vector database in Notebook 2  
can also retrieve based on occasion and vibe if needed.

In [23]:
def enrich_metadata(row: pd.Series) -> str:
    """
    Appends classification labels to the existing restaurant_metadata string.
    Original metadata is preserved; new labels are appended.
    """
    base = str(row['restaurant_metadata']).rstrip('.')
    return (
        f"{base} "
        f"Cuisine Group: {row['simple_cuisine_group']}. "
        f"Dining Format: {row['dining_format']}. "
        f"Best For: {row['predicted_occasion']}. "
        f"Vibe: {row['predicted_vibe']}."
    )

restaurants['restaurant_metadata'] = restaurants.apply(enrich_metadata, axis=1)

print("✅ restaurant_metadata enriched with classification labels.")
print("\nSample enriched metadata:")
print(f"\n[0] {restaurants.loc[0, 'Name']}:")
print(f"    {restaurants.loc[0, 'restaurant_metadata']}")
print(f"\n[13] {restaurants.loc[13, 'Name']}:")
print(f"    {restaurants.loc[13, 'restaurant_metadata']}")

✅ restaurant_metadata enriched with classification labels.

Sample enriched metadata:

[0] Somni:
    Somni is a Spanish Modernist restaurant located in West Hollywood, Los Angeles. A 14-seat Spanish Modernist chef's counter offering a 20+ course tasting menu led by Chef Aitor Zabala with one seating per night. Price range: $$$$$. Atmosphere: Fine-Dining. Michelin Guide: Michelin 3-Star. Customer Rating: 5.0/5 Cuisine Group: Spanish. Dining Format: Tasting Menu / Omakase. Best For: Special Occasion. Vibe: Refined & Elegant.

[13] Holbox:
    Holbox is a Mexican / Seafood restaurant located in South Los Angeles, Los Angeles. A Yucatecan-style seafood counter from Chef Gilberto Cetina Jr. inside Mercado La Paloma offering some of LA's most delicious and affordable Michelin-starred food. Price range: $$. Atmosphere: Casual. Michelin Guide: Michelin 1-Star. Customer Rating: 5.0/5 Cuisine Group: Seafood. Dining Format: A La Carte / Casual. Best For: Business Dining. Vibe: Cozy & Relaxed.


## 💾 Cell 24 — Export Final Classified Dataset

Saves the fully enriched dataset. Confidence score columns are retained for  
debugging but can be dropped in a production export if desired.

In [24]:
OUTPUT_PATH = "../data/restaurants_with_classifications.csv"

restaurants.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Classified dataset saved to: {OUTPUT_PATH}")
print(f"   Rows    : {len(restaurants)}")
print(f"   Columns : {len(restaurants.columns)}")
print(f"\nFinal columns:")
for i, col in enumerate(restaurants.columns, 1):
    is_new = col in ['simple_cuisine_group', 'dining_format', 
                     'predicted_occasion', 'occasion_confidence',
                     'predicted_vibe', 'vibe_confidence']
    tag = " ← NEW" if is_new else ""
    print(f"   {i:>2}. {col}{tag}")

✅ Classified dataset saved to: ../data/restaurants_with_classifications.csv
   Rows    : 94
   Columns : 21

Final columns:
    1. Name
    2. Location
    3. Description
    4. Address
    5. Telephone Number
    6. Price
    7. Cuisine Type
    8. Dining Atmosphere
    9. Sky-High Rooftop
   10. Michelin-Guide
   11. Customer Ratings
   12. Operation Hours
   13. Reservations
   14. Dress Code
   15. restaurant_metadata
   16. simple_cuisine_group ← NEW
   17. dining_format ← NEW
   18. predicted_occasion ← NEW
   19. occasion_confidence ← NEW
   20. predicted_vibe ← NEW
   21. vibe_confidence ← NEW


## 📊 Cell 25 — Final Dataset Summary Report

In [ ]:
print("=" * 62)
print("   CLASSIFICATION PIPELINE — FINAL SUMMARY")
print("=" * 62)
print(f"  Total Restaurants         : {len(restaurants)}")
print(f"  New Classification Columns: 4")
print()

print("  Layer 1 — simple_cuisine_group (rule-based):")
for g, c in restaurants['simple_cuisine_group'].value_counts().items():
    print(f"    {g:<28}: {c}")
print()

print("  Layer 2 — dining_format (keyword-based):")
for f, c in restaurants['dining_format'].value_counts().items():
    print(f"    {f:<32}: {c}")
print()

print("  Layer 3 — predicted_occasion (zero-shot LLM):")
for o, c in restaurants['predicted_occasion'].value_counts().items():
    print(f"    {o:<32}: {c}")
print(f"    Avg confidence            : {restaurants['occasion_confidence'].mean():.3f}")
print()

print("  Layer 4 — predicted_vibe (zero-shot LLM):")
for v, c in restaurants['predicted_vibe'].value_counts().items():
    print(f"    {v:<32}: {c}")
print(f"    Avg confidence            : {restaurants['vibe_confidence'].mean():.3f}")
print()

print(f"  Output file               : restaurants_with_classifications.csv")
print("=" * 62)
print("  ✅ All 94 restaurants classified across 4 dimensions!")
print("=" * 62)
print()
print("🚀 Next step: Notebook 4 — Gradio Dashboard UI")
print("   Use restaurants_with_classifications.csv as input")
print("   New filter options to add in Gradio:")
print("     • Cuisine Group  (simple_cuisine_group)")
print("     • Dining Format  (dining_format)")
print("     • Occasion       (predicted_occasion)")
print("     • Vibe           (predicted_vibe)")

   CLASSIFICATION PIPELINE — FINAL SUMMARY
  Total Restaurants         : 94
  New Classification Columns: 4

  Layer 1 — simple_cuisine_group (rule-based):
    Japanese                    : 14
    Contemporary American       : 12
    Italian                     : 12
    Asian Fusion                : 11
    Steakhouse                  : 11
    French                      : 4
    Seafood                     : 3
    Japanese Fusion             : 3
    Spanish                     : 2
    Mexican / Latin             : 2
    American Casual             : 1

  Layer 2 — dining_format (keyword-based):
    Full Service                    : 37
    A La Carte / Casual             : 37
    Tasting Menu / Omakase          : 20

  Layer 3 — predicted_occasion (zero-shot LLM):
    Special Occasion                : 69
    Foodie Adventure                : 16
    Casual Night Out                : 8
    Business Dining                 : 1
    Avg confidence            : 0.359

  Layer 4 — predicted_vibe